In [1]:
import re
import json
import time
from pathlib import Path
from typing import List, Dict, Optional, Set

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# --- Config ---
OUTPUT_DIR = Path("../pokemon_big_data_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
BULBA_API = "https://bulbapedia.bulbagarden.net/w/api.php"
POKEAPI = "https://pokeapi.co/api/v2"

def get_session():
    s = requests.Session()
    retries = Retry(total=5, backoff_factor=1, status_forcelist=[502, 503, 504])
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.headers.update({"User-Agent": "PokemonBigDataBot/2.0 (Bachelor Project)"})
    return s

session = get_session()

_TITLE_EXISTS_CACHE = {}


def page_exists(title: str) -> bool:
    if title in _TITLE_EXISTS_CACHE:
        return _TITLE_EXISTS_CACHE[title]

    try:
        resp = session.get(BULBA_API, params={
            "action": "query",
            "titles": title,
            "format": "json",
            "formatversion": 2
        }).json()
        pages = resp.get("query", {}).get("pages", [])
        exists = bool(pages) and not pages[0].get("missing", False)
    except Exception:
        exists = False

    _TITLE_EXISTS_CACHE[title] = exists
    return exists


def resolve_existing_root_title(candidate_titles: List[str]) -> Optional[str]:
    for title in candidate_titles:
        if title and page_exists(title):
            return title
    return None

import re
from typing import Dict, Optional, Set

class LocationMapper:
    """
    Final optimized Mapper for Pokemon Big Data Walkthrough.
    Handles specific Kanto Underground Path edge cases and generic placeholders.
    """
    def __init__(self, api_response: Dict):
        # 1. Valid Slugs from PokeAPI (via your limit=2000 call)
        self.valid_slugs = {item['name'] for item in api_response['results']}
        self.cache = {}
        self.misses = set()

        # 2. Refined Blacklist
        # Added 'cave' to ignore the generic link
        self.blacklist = [
            "edit section's source code", "file:", "badge", "pharmacy",
            "center", "mart", "interior", "entrance", "style", "sushi high roller",
            "house", "gate", "link trade", "day care", "hall of fame", "cave"
        ]

        # 3. Targeted Overrides
        # Handled the 'Underground Path' links specifically
        self.hard_map = {
            # Underground Paths (No wild encounters in PokeAPI, mapped to nearest route if needed)
            "underground-path-kanto-routes-5-6": "kanto-route-5",
            "underground-path-kanto-routes-7-8": "kanto-route-7",

            # Existing Sea Route & Special Fixes
            "sinnoh-route-220": "sinnoh-sea-route-220",
            "sinnoh-route-223": "sinnoh-sea-route-223",
            "sinnoh-route-226": "sinnoh-sea-route-226",
            "sinnoh-route-230": "sinnoh-sea-route-230",
            "kanto-route-19": "kanto-sea-route-19",
            "kanto-route-20": "kanto-sea-route-20",
            "kanto-route-21": "kanto-sea-route-21",
            "johto-route-40": "johto-sea-route-40",
            "johto-route-41": "johto-sea-route-41",
            "digletts-cave": "digletts-cave",
            "challengers-cave": "challengers-cave",
            "mt-moon-square": "mt-moon",
            "ilex-forest-shrine": "ilex-forest",
            "victory-road": "kanto-victory-road-1"
        }

    def _clean_title(self, title: str) -> str:
        """Strips wiki metadata and parenthetical qualifiers."""
        t = re.sub(r"(?i)edit section's source code:\s*", "", title)
        # We KEEP parentheses temporarily for Underground Paths, but remove for others
        if "Underground Path" not in t:
            t = re.sub(r"\(.*?\)", "", t)
        return t.strip()

    def _slugify(self, text: str) -> str:
        """Standardizes text to PokeAPI slug format."""
        s = text.lower().replace("é", "e").replace("’", "").replace("'", "")
        s = re.sub(r"[^a-z0-9]+", "-", s)
        return s.strip("-")

    def resolve(self, raw_title: str, route_prefix: str) -> Optional[str]:
        # Step 1: Filter generic 'Cave' or junk
        if raw_title.lower().strip() == "cave" or any(bad in raw_title.lower() for bad in self.blacklist):
            return None

        # Step 2: Cache check
        if raw_title in self.cache:
            return self.cache[raw_title]

        # Step 3: Clean and Slugify
        clean_name = self._clean_title(raw_title)

        # Step 4: Route Logic
        route_match = re.search(r"Route\s+(\d+)", clean_name, re.I)
        if route_match:
            r_num = route_match.group(1)
            if "Kanto" in raw_title: slug = f"kanto-route-{r_num}"
            elif "Johto" in raw_title: slug = f"johto-route-{r_num}"
            else: slug = f"{route_prefix}-{r_num}"
        else:
            slug = self._slugify(clean_name)

        # Step 5: Hard Mapping (Crucial for the Underground Paths)
        slug = self.hard_map.get(slug, slug)

        # Step 6: Final Validation
        if slug in self.valid_slugs:
            self.cache[raw_title] = slug
            return slug

        # Log any remaining failures
        if clean_name.strip():
            self.misses.add(f"{raw_title} -> tried: {slug}")

        return None
# 1. Fetch the data once
response = requests.get("https://pokeapi.co/api/v2/location?limit=2000").json()

# 2. Initialize the Mapper with that JSON
mapper = LocationMapper(response)
print("✅ Refined Mapper Ready.")

✅ Refined Mapper Ready.


In [2]:
def get_walkthrough_parts(root_title: str) -> List[Dict]:
    params = {"action": "parse", "page": root_title, "prop": "text", "format": "json", "formatversion": 2}
    resp = session.get(BULBA_API, params=params).json()
    page_html = resp.get("parse", {}).get("text")
    if not page_html:
        return []

    soup = BeautifulSoup(page_html, "lxml")

    parts = []
    for a in soup.select("div.mw-parser-output a[href*='/Part']"):
        title = a.get("title")
        if title and root_title in title:
            m = re.search(r"Part[_ ](\d+)", title)
            if m: parts.append({"part": int(m.group(1)), "title": title})
    return sorted(parts, key=lambda x: x['part'])

def extract_game_data(game_cfg: Dict):
    candidate_titles = game_cfg.get("candidate_root_titles", [game_cfg["root_title"]])
    resolved_root_title = resolve_existing_root_title(candidate_titles)
    if not resolved_root_title:
        print(f"⚠️ No valid walkthrough title found for {game_cfg['game_key']}. Candidates: {candidate_titles}")
        return pd.DataFrame()

    parts = get_walkthrough_parts(resolved_root_title)
    cumulative_slugs = []
    boss_encounters = []

    # Build regex for Boss detection
    tokens = ["Gym", "Elite Four", "Champion"] + game_cfg.get("gym_leaders", game_cfg.get("bosses", []))
    boss_re = re.compile(r"(?i)\b(" + "|".join(map(re.escape, tokens)) + r")\b")

    for part in tqdm(parts, desc=f"Processing {game_cfg['game_key']}"):
        resp = session.get(BULBA_API, params={
            "action": "parse", "page": part["title"], "prop": "text", "format": "json", "formatversion": 2
        }).json()

        soup = BeautifulSoup(resp["parse"]["text"], "lxml")
        content = soup.find("div", class_="mw-parser-output")

        for el in content.find_all(['h2', 'h3', 'a']):
            if el.name == 'a' and el.get('title'):
                # Heuristic for location links
                if any(k in el['title'] for k in ["Route", "City", "Cave", "Forest", "Mt."]):
                    slug = mapper.resolve(el['title'], game_cfg["route_prefix"])
                    if slug: cumulative_slugs.append(slug)

            elif el.name in ['h2', 'h3'] and boss_re.search(el.get_text()):
                # Capture snapshot
                current_reachable = list(dict.fromkeys(cumulative_slugs))
                boss_encounters.append({
                    "game": game_cfg["game_key"],
                    "boss_name": el.get_text().strip(),
                    "part": part["part"],
                    "reachable_locations": current_reachable,
                    "location_count": len(current_reachable)
                })

    return pd.DataFrame(boss_encounters)

In [3]:
BASE_GAME_GROUPS = [
    {
        "versions": ["red", "blue"],
        "root_title": "Walkthrough:Pokémon Red and Blue",
        "version_root_titles": {
            "red": ["Walkthrough:Pokémon Red Version", "Walkthrough:Pokémon Red"],
            "blue": ["Walkthrough:Pokémon Blue Version", "Walkthrough:Pokémon Blue"]
        },
        "route_prefix": "kanto-route",
        "bosses": ["Brock", "Misty", "Lt. Surge", "Erika", "Koga", "Sabrina", "Blaine", "Giovanni", "Lorelei", "Bruno", "Agatha", "Lance", "Blue"]
    },
    {
        "versions": ["gold", "silver"],
        "root_title": "Walkthrough:Pokémon Gold and Silver",
        "version_root_titles": {
            "gold": ["Walkthrough:Pokémon Gold Version", "Walkthrough:Pokémon Gold"],
            "silver": ["Walkthrough:Pokémon Silver Version", "Walkthrough:Pokémon Silver"]
        },
        "route_prefix": "johto-route",
        "bosses": ["Falkner", "Bugsy", "Whitney", "Morty", "Chuck", "Jasmine", "Pryce", "Clair", "Will", "Koga", "Bruno", "Karen", "Lance"]
    },
    {
        "versions": ["ruby", "sapphire"],
        "root_title": "Walkthrough:Pokémon Ruby and Sapphire",
        "version_root_titles": {
            "ruby": ["Walkthrough:Pokémon Ruby Version", "Walkthrough:Pokémon Ruby"],
            "sapphire": ["Walkthrough:Pokémon Sapphire Version", "Walkthrough:Pokémon Sapphire"]
        },
        "route_prefix": "hoenn-route",
        "bosses": ["Roxanne", "Brawly", "Wattson", "Flannery", "Norman", "Winona", "Tate and Liza", "Wallace", "Sidney", "Phoebe", "Glacia", "Drake", "Steven"]
    },
    {
        "versions": ["diamond", "pearl"],
        "root_title": "Walkthrough:Pokémon Diamond and Pearl",
        "version_root_titles": {
            "diamond": ["Walkthrough:Pokémon Diamond Version", "Walkthrough:Pokémon Diamond"],
            "pearl": ["Walkthrough:Pokémon Pearl Version", "Walkthrough:Pokémon Pearl"]
        },
        "route_prefix": "sinnoh-route",
        "bosses": ["Roark", "Gardenia", "Maylene", "Crasher Wake", "Fantina", "Byron", "Candice", "Volkner", "Aaron", "Bertha", "Flint", "Lucian", "Cynthia"]
    },
    {
        "versions": ["black", "white"],
        "root_title": "Walkthrough:Pokémon Black and White",
        "version_root_titles": {
            "black": ["Walkthrough:Pokémon Black Version", "Walkthrough:Pokémon Black"],
            "white": ["Walkthrough:Pokémon White Version", "Walkthrough:Pokémon White"]
        },
        "route_prefix": "unova-route",
        "bosses": ["Cilan", "Lenora", "Burgh", "Elesa", "Clay", "Skyla", "Brycen", "Drayden", "Shauntal", "Grimsley", "Caitlin", "Marshal", "N", "Ghetsis", "Alder"]
    },
    {
        "versions": ["x", "y"],
        "root_title": "Walkthrough:Pokémon X and Y",
        "version_root_titles": {
            "x": ["Walkthrough:Pokémon X"],
            "y": ["Walkthrough:Pokémon Y"]
        },
        "route_prefix": "kalos-route",
        "bosses": ["Viola", "Grant", "Korrina", "Ramos", "Clemont", "Valerie", "Olympia", "Wulfric", "Malva", "Siebold", "Wikstrom", "Drasna", "Diantha"]
    }
]

GAMES_CONFIG = []
for group in BASE_GAME_GROUPS:
    for version in group["versions"]:
        version_titles = group.get("version_root_titles", {}).get(version, [])
        GAMES_CONFIG.append({
            "game_key": version,
            "root_title": group["root_title"],
            "candidate_root_titles": version_titles + [group["root_title"]],
            "route_prefix": group["route_prefix"],
            "bosses": group["bosses"]
        })

print(f"✅ Configured {len(GAMES_CONFIG)} individual game versions.")

all_results = []
for config in GAMES_CONFIG:
    df = extract_game_data(config)
    all_results.append(df)

    # Intermediate Save
    df.to_json(OUTPUT_DIR / f"{config['game_key']}_data.jsonl", orient="records", lines=True)

final_df = pd.concat(all_results, ignore_index=True)
print(f"📊 Extraction Complete! Total Boss Fights Captured: {len(final_df)}")

✅ Configured 12 individual game versions.


Processing y: 100%|██████████| 17/17 [00:03<00:00,  4.91it/s]

📊 Extraction Complete! Total Boss Fights Captured: 180


In [4]:
# Save the 'Mapping Misses' for manual review
with open(OUTPUT_DIR / "unmapped_locations.json", "w") as f:
    json.dump(list(mapper.misses), f, indent=4)

print(f"⚠️ {len(mapper.misses)} locations were not found in PokéAPI. Check 'unmapped_locations.json'")

# Quick Preview of the Progression
final_df[['game', 'boss_name', 'location_count']].head(20)

⚠️ 0 locations were not found in PokéAPI. Check 'unmapped_locations.json'


,game,boss_name,location_count
0,red,Blue's house[edit source],5
1,red,Pewter Gym[edit source],8
2,red,"Gym Badges, Explained[edit source]",14
3,red,Cerulean Gym[edit source],14
4,red,Vermilion Gym[edit source],19
5,red,Celadon Gym[edit source],25
6,red,Saffron Gym[edit source],25
7,red,Fuchsia Gym[edit source],30
8,red,Cinnabar Gym[edit source],33
9,red,Viridian Gym[edit source],33


In [5]:
def get_location_areas(location_slugs: List[str]) -> Dict[str, List[str]]:
    """
    Maps each Location slug to its respective Location Area slugs.
    Example: 'kanto-route-1' -> ['kanto-route-1-area']
    """
    area_map = {}
    unique_locations = list(set(location_slugs))

    print(f"🌐 Fetching areas for {len(unique_locations)} unique locations...")

    for slug in tqdm(unique_locations, desc="Mapping Areas"):
        try:
            # PokeAPI Location endpoint returns a list of 'areas'
            url = f"{POKEAPI}/location/{slug}"
            resp = session.get(url, timeout=5)

            if resp.status_code == 200:
                data = resp.json()
                # Extract the names of all areas in this location
                areas = [area['name'] for area in data.get('areas', [])]
                area_map[slug] = areas
            else:
                area_map[slug] = []

        except Exception as e:
            print(f"⚠️ Error fetching areas for {slug}: {e}")
            area_map[slug] = []

        # Respectful delay for Big Data mass-requests
        time.sleep(0.1)

    return area_map

# --- Enrichment Step ---
# 1. Collect all unique slugs from your previous results
all_slugs = []
for file in Path("../pokemon_big_data_outputs").glob("*.jsonl"):
    df = pd.read_json(file, lines=True)
    for loc_list in df['reachable_locations']:
        all_slugs.extend(loc_list)

# 2. Map them to Areas
master_area_map = get_location_areas(all_slugs)

# 3. Save the Map (Transparency)
with open(OUTPUT_DIR / "location_to_area_map.json", "w") as f:
    json.dump(master_area_map, f, indent=4)

print(f"✅ Area mapping complete. Saved to location_to_area_map.json")

🌐 Fetching areas for 189 unique locations...


Mapping Areas: 100%|██████████| 189/189 [00:25<00:00,  7.32it/s]

✅ Area mapping complete. Saved to location_to_area_map.json
